# Critical Input DEQN: Commitment

This notebook trains the commitment optimal-policy DEQN network. The state is augmented with promise variables attached to forward-looking implementability constraints.

In [ ]:
# Configure paths and commitment-policy training settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT = ARTIFACT_ROOT / 'commitment'
OUT.mkdir(parents=True, exist_ok=True)

STEPS = 50_000
QMC_TRAIN = 512
QMC_VAL = 1024
N_VAL_STATES = 2048
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
PROMISE_INIT_SCALE = 1.0
LOG_EVERY = 100
TARGET_RMS = None
TARGET_MAX_ABS = None
EARLY_STOP_PATIENCE = None
MIN_STEPS_BEFORE_STOP = None
STOP_VAL_STATES = 512
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'
print(ROOT)
print(OUT)

In [ ]:
# Train the commitment network with nonzero promise initialization.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_optimal',
    '--output-dir', str(OUT),
    '--kind', 'commitment',
    '--steps', str(STEPS),
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--promise-init-scale', str(PROMISE_INIT_SCALE),
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
]
if TARGET_RMS is not None:
    cmd += ['--target-rms', str(TARGET_RMS)]
if TARGET_MAX_ABS is not None:
    cmd += ['--target-max-abs', str(TARGET_MAX_ABS)]
if EARLY_STOP_PATIENCE is not None:
    cmd += ['--early-stop-patience', str(EARLY_STOP_PATIENCE)]
if MIN_STEPS_BEFORE_STOP is not None:
    cmd += ['--min-steps-before-stop', str(MIN_STEPS_BEFORE_STOP)]
subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
# Inspect out-of-sample residual diagnostics for commitment.
with (OUT / 'commitment_eval.json').open('r', encoding='utf-8') as fh:
    commitment_eval = json.load(fh)
commitment_eval